# 09 -- Torsion and drift

Floor-4 torsion pair, interstory drift and base rocking (windowed).

In [1]:
%matplotlib inline

import numpy as np
from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

# The example data folder (edit to your path if needed).
DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"

# A short window keeps every example fast (one ~10 min file).
WSTART, WLEN = "2026-05-31 15:00:00", "10min"

In [2]:
ds = SensorDataset(DATA, verbose=True)
ds.devices

------------------------------------------------------------
SensorDataset
------------------------------------------------------------
path        : C:\Users\ppala\Desktop\02_31MAY2026
files       : 32
time span   : 2026-05-31 14:52:12  ->  2026-05-31 20:02:13
duration    : 18601.0 s
devices     : MNAT0031, MNAT0034, MOF00134, MOF00135, MOF00136
fs / dt     : 252.5885 Hz / 0.003959 s
------------------------------------------------------------
axes (per sensor):
  MNAT0031   -> (3, 1, 5)
  MNAT0034   -> (3, 1, 5)
  MOF00134   -> (0, 1, 2)
  MOF00135   -> (3, 1, 5)
  MOF00136   -> (3, 1, 5)
------------------------------------------------------------
on-disk size: 782.17 MB
RAM         : used 32.30 GB / avail 1.21 GB (96%)
------------------------------------------------------------


['MNAT0031', 'MNAT0034', 'MOF00134', 'MOF00135', 'MOF00136']

In [3]:
# Sensor geometry up front (approximate UTM E/N, elevation). Building methods
# (torsion, mode shapes, drift) need these references.
settings.SENSOR_GEOMETRY["MOF00134"].update(E=500000.0, N=6300000.0, elev=0.0)
settings.SENSOR_GEOMETRY["MNAT0031"].update(E=500000.0, N=6300000.0, elev=7.0)
settings.SENSOR_GEOMETRY["MNAT0034"].update(E=500000.0, N=6300000.0, elev=10.5)
settings.SENSOR_GEOMETRY["MOF00135"].update(E=500000.0, N=6300000.0, elev=14.0)
settings.SENSOR_GEOMETRY["MOF00136"].update(E=500006.0, N=6300004.0, elev=14.0)
{d: settings.SENSOR_GEOMETRY[d]["floor"] for d in settings.SENSOR_GEOMETRY}

{'MOF00134': -1, 'MNAT0031': 2, 'MNAT0034': 3, 'MOF00135': 4, 'MOF00136': 4}

## Floor-4 torsion (MOF00135 + MOF00136)

In [4]:
from asdea_sensors.building import torsion, geometry, drift, base_rocking
a = ds.MOF00135.window(WSTART, WLEN).filter(0.1, 24.9).signal(components="x")
b = ds.MOF00136.window(WSTART, WLEN).filter(0.1, 24.9).signal(components="x")
n = min(a.n, b.n)
dist = geometry.plan_distance(settings.SENSOR_GEOMETRY, "MOF00135", "MOF00136")
theta = torsion.floor_rotation(a.acc_x[:n], b.acc_x[:n], dist)
spec = torsion.torsional_spectrum(theta, a.dt, smooth=None)
print("plan distance:", round(dist,2), "m  | torsional freq:", round(float(spec["torsional_freq"]),2), "Hz")

[signal] MOF00135 n=149240 dt=0.004021 comps=x
[signal] MOF00136 n=150500 dt=0.003987 comps=x
plan distance: 7.21 m  | torsional freq: 0.2 Hz


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MOF00135' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(
C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\derive\filters.py:49: UserWarning: obspy not available; falling back to the scipy engine
  warnings.warn("obspy not available; falling back to the scipy engine")
C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MOF00136' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


## Interstory drift (floor 4 vs floor 3)

In [5]:
up = ds.MOF00135.window(WSTART, WLEN).filter(0.1, 24.9).derive().signal(components="x")
lo = ds.MNAT0034.window(WSTART, WLEN).filter(0.1, 24.9).derive().signal(components="x")
n = min(up.n, lo.n)
d = drift.interstory_drift(up.disp_x[:n], lo.disp_x[:n], story_height=3.5)
print("max drift:", round(float(d["max_drift"]),6), "m | max ratio:", round(float(d["max_ratio"]),6))

[signal] MOF00135 n=149240 dt=0.004021 comps=x


[signal] MNAT0034 n=151320 dt=0.003966 comps=x
max drift: 0.088495 m | max ratio: 0.025284


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MNAT0034' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


## Base rocking

In [6]:
base = ds.MOF00134.window(WSTART, WLEN).filter(0.1, 24.9).signal(components="z")
rock = base_rocking.compute(base.acc_z, base.dt)
print("rocking freq:", round(float(rock["rocking_freq"]),2), "Hz")

[signal] MOF00134 n=149340 dt=0.004018 comps=z
rocking freq: 12.44 Hz
